# Tier 2: Improving Article Credibility Assessment

The experiments that led from the binary v1 model to the v2 objective. Every number is read from files written by the scripts in `experiments_tier2/`.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd()
OUTP = ROOT / "outputs"
DATA = ROOT / "datasets" / "processed_tier2"

train = pd.read_csv(DATA / "train.csv")
val   = pd.read_csv(DATA / "validation.csv")
test  = pd.read_csv(DATA / "test.csv")
print(f"train {len(train)}   validation {len(val)}   test {len(test)}")

train 1483   validation 318   test 318


## 1. The diagnosis: what the binary label is

`NOT_CREDIBLE` is a star rating of 2 or less, and the rating is close to a sum over the ten criteria.

In [2]:
CRITERIA = ["costs", "benefits", "harms", "evidence_quality", "disease_mongering",
            "independent_srcs", "alternatives", "availability", "novelty", "press_release"]
F = [f"fail_{c}" for c in CRITERIA]

print("binary_label is exactly rating <= 2:")
print(pd.crosstab(train["rating"], train["binary_label"]))

n_failed = train[F].fillna(0).sum(axis=1)
n_valid  = train[F].notna().sum(axis=1)
frac_failed = n_failed / n_valid.clip(lower=1)

print(f"\ncorr(rating, fraction of criteria failed) = {np.corrcoef(train['rating'], frac_failed)[0,1]:+.3f}")
print(f"mean criteria actually annotated per article = {n_valid.mean():.2f}")

binary_label is exactly rating <= 2:
binary_label  CREDIBLE  NOT_CREDIBLE
rating                              
0                    0            22
1                    0           123
2                    0           352
3                  473             0
4                  361             0
5                  152             0

corr(rating, fraction of criteria failed) = -0.944
mean criteria actually annotated per article = 8.87


### Training signals per article

In [3]:
print(f"binary labels          : {len(train):>6,} signals")
print(f"ordinal comparisons    : {len(train)*5:>6,} signals")
print(f"criteria judgements    : {int(train[F].notna().sum().sum()):>6,} signals")
print(f"\ntotal available        : {len(train) + len(train)*5 + int(train[F].notna().sum().sum()):>6,}")
print(f"actually used by the shipped model: {len(train):,}")

binary labels          :  1,483 signals
ordinal comparisons    :  7,415 signals
criteria judgements    : 13,153 signals

total available        : 22,051
actually used by the shipped model: 1,483


## 2. Experiment 1: the training objective

Six combinations of the binary, ordinal (CORAL) and criteria heads, on PubMedBERT with 512 tokens, 3 seeds each.

In [4]:
allr = pd.read_csv(OUTP / "tier2_lab_all_rules.csv")
rate = allr[(allr.rule == "rate") & (allr.seed != "ENS")]

obj = rate[rate.variant.str.match(r"^[A-F]_")]
tbl = (obj.groupby(["variant", "score"])[["accuracy", "f1_macro", "recall_notcred", "roc_auc"]]
         .agg(["mean", "std"]).round(4))
tbl.columns = ["_".join(c) for c in tbl.columns]
tbl = tbl.sort_values("f1_macro_mean", ascending=False)
print(tbl[["f1_macro_mean", "f1_macro_std", "accuracy_mean",
           "recall_notcred_mean", "roc_auc_mean"]].to_string())

                   f1_macro_mean  f1_macro_std  accuracy_mean  recall_notcred_mean  roc_auc_mean
variant     score                                                                               
E_ord_crit  blend         0.6883        0.0316         0.7285               0.5483        0.7221
            ord           0.6863        0.0295         0.7275               0.5421        0.7222
C_criteria  crit          0.6843        0.0204         0.7306               0.5171        0.7232
B_ordinal   ord           0.6816        0.0141         0.7243               0.5327        0.7119
E_ord_crit  crit          0.6799        0.0129         0.7222               0.5327        0.7222
F_all_heads ord           0.6697        0.0046         0.7159               0.5078        0.7095
            blend         0.6645        0.0110         0.7107               0.5047        0.7123
D_bin_crit  crit          0.6614        0.0156         0.7086               0.4985        0.6979
F_all_heads crit          0.65

### With and without the binary head

In [5]:
pairs = [("C_criteria", "D_bin_crit", "crit"), ("E_ord_crit", "F_all_heads", "ord")]
for without, with_, sc in pairs:
    a = rate[(rate.variant == without) & (rate.score == sc)]
    b = rate[(rate.variant == with_)  & (rate.score == sc)]
    print(f"{without:<14} {sc:<5} macroF1={a.f1_macro.mean():.4f}   AUC={a.roc_auc.mean():.4f}")
    print(f"{with_:<14} {sc:<5} macroF1={b.f1_macro.mean():.4f}   AUC={b.roc_auc.mean():.4f}"
          f"   <- adding the binary head: {b.f1_macro.mean()-a.f1_macro.mean():+.4f}\n")

C_criteria     crit  macroF1=0.6843   AUC=0.7232
D_bin_crit     crit  macroF1=0.6614   AUC=0.6979   <- adding the binary head: -0.0228

E_ord_crit     ord   macroF1=0.6863   AUC=0.7222
F_all_heads    ord   macroF1=0.6697   AUC=0.7095   <- adding the binary head: -0.0166



## 3. Experiment 2: head and tail truncation

The first 384 tokens plus the last 128, against plain 512 token truncation.

In [6]:
g = rate[rate.variant == "G_headtail"].groupby("score")[["f1_macro", "roc_auc"]].mean().round(4)
e = rate[rate.variant == "E_ord_crit"].groupby("score")[["f1_macro", "roc_auc"]].mean().round(4)
cmp = e.join(g, lsuffix="_plain512", rsuffix="_headtail")
print(cmp.to_string())
print("\nhead+tail truncation is worse on every score.")

       f1_macro_plain512  roc_auc_plain512  f1_macro_headtail  roc_auc_headtail
score                                                                          
blend             0.6883            0.7221             0.6609            0.6972
crit              0.6799            0.7222             0.6457            0.6963
ord               0.6863            0.7222             0.6615            0.6961

head+tail truncation is worse on every score.


## 4. Experiment 3: other encoders

ModernBERT (2,048 tokens) and BiomedBERT-large against PubMedBERT-base.

In [7]:
enc = rate[rate.variant.str.match(r"^[H-J]_")]
if len(enc):
    t = (enc.groupby(["variant", "score"])[["f1_macro", "roc_auc"]]
           .agg(["mean", "count"]).round(4))
    t.columns = ["_".join(c) for c in t.columns]
    print(t.sort_values("roc_auc_mean", ascending=False).to_string())
base = rate[(rate.variant == "E_ord_crit")].groupby("score")[["f1_macro","roc_auc"]].mean().round(4)
print("\nPubMedBERT-base reference (E_ord_crit):")
print(base.to_string())

                         f1_macro_mean  f1_macro_count  roc_auc_mean  roc_auc_count
variant       score                                                                
J_biomedlarge crit              0.6645               1        0.7346              1
              blend             0.6563               1        0.7340              1
              crit_nodm         0.6653               1        0.7340              1
              crit_lr           0.6635               1        0.7302              1
              ord               0.6399               1        0.7300              1
I_modernbert  ord               0.6479               1        0.6937              1
              blend             0.6344               1        0.6894              1
              crit_nodm         0.5960               1        0.6740              1
              crit              0.5934               1        0.6722              1
              crit_lr           0.5978               1        0.6721        

## 5. Is the improvement real? Eight seed confirmation

The baseline and the best variant were each retrained 8 times and compared with a Welch t-test.

In [8]:
conf = pd.read_csv(OUTP / "tier2_confirm_seeds.csv")
summ = (conf.groupby(["config", "score"])
            .agg(n=("f1_macro", "size"), macroF1=("f1_macro", "mean"),
                 sd=("f1_macro", "std"), acc=("accuracy", "mean"),
                 recall=("recall_notcred", "mean"), auc=("roc_auc", "mean"))
            .round(4))
summ["SE"] = (summ["sd"] / np.sqrt(summ["n"])).round(4)
print(summ.to_string())

from scipy import stats
a = conf[(conf.config == "BASE_binary") & (conf.score == "bin")].f1_macro.values
print("\nWelch t-test vs the binary baseline (per-seed macro-F1):")
for s in sorted(conf[conf.config == "BEST_ord_crit"].score.unique()):
    b = conf[(conf.config == "BEST_ord_crit") & (conf.score == s)].f1_macro.values
    t, p = stats.ttest_ind(b, a, equal_var=False)
    flag = "  <-- significant" if p < 0.05 else ""
    print(f"  {s:<10} delta={b.mean()-a.mean():+.4f}   t={t:+.2f}   p={p:.4f}{flag}")

                         n  macroF1      sd     acc  recall     auc      SE
config        score                                                        
BASE_binary   bin        8   0.6582  0.0243  0.7036  0.5058  0.7009  0.0086
BEST_ord_crit crit       8   0.6798  0.0203  0.7198  0.5444  0.7191  0.0072
              crit_lr    8   0.6851  0.0160  0.7229  0.5596  0.7179  0.0057
              crit_nodm  8   0.6812  0.0173  0.7209  0.5467  0.7173  0.0061
              ord        8   0.6738  0.0160  0.7142  0.5397  0.7193  0.0057



Welch t-test vs the binary baseline (per-seed macro-F1):
  crit       delta=+0.0216   t=+1.93   p=0.0746
  crit_lr    delta=+0.0270   t=+2.62   p=0.0221  <-- significant
  crit_nodm  delta=+0.0230   t=+2.18   p=0.0486  <-- significant
  ord        delta=+0.0157   t=+1.53   p=0.1525


## 6. Does it explain better? Per criterion quality

In [9]:
newp = OUTP / "tier2_v2_criteria_performance.csv"
old = pd.read_csv(OUTP / "tier2_criteria_performance.csv")
if newp.exists():
    new = pd.read_csv(newp)
    key = "criterion"
    oldf = old.set_index(key)["f1"] if "f1" in old.columns else None
    cmp = new.set_index(key)[["f1", "mcc", "auc", "base_rate", "pred_rate"]].copy()
    if oldf is not None:
        cmp["f1_old"] = oldf
        cmp["delta_f1"] = (cmp["f1"] - cmp["f1_old"]).round(3)
    print(cmp.sort_values("f1", ascending=False).to_string())
    print(f"\nmean criteria F1 : {new['f1'].mean():.3f}")
    print(f"mean criteria MCC: {new['mcc'].mean():.3f}   (MCC is the honest one -- "
          f"F1 can be gamed by a constant classifier)")
else:
    print("final model not built yet; run confirm_over_8_seeds.py --mode final")

                      f1    mcc    auc  base_rate  pred_rate  f1_old  delta_f1
criterion                                                                     
costs              0.848  0.314  0.751      0.779      0.779   0.811     0.037
harms              0.774  0.300  0.724      0.677      0.677   0.730     0.044
benefits           0.741  0.251  0.678      0.655      0.655   0.697     0.044
evidence_quality   0.696  0.240  0.696      0.601      0.601   0.703    -0.007
independent_srcs   0.588  0.193  0.600      0.489      0.489   0.487     0.101
alternatives       0.573  0.121  0.586      0.515      0.515   0.592    -0.019
novelty            0.441  0.197  0.650      0.304      0.304   0.394     0.047
press_release      0.423  0.344  0.731      0.120      0.120   0.391     0.032
availability       0.348  0.126  0.610      0.254      0.254   0.319     0.029
disease_mongering  0.125  0.024  0.538      0.104      0.104   0.108     0.017

mean criteria F1 : 0.556
mean criteria MCC: 0.211  